# DNA–Lipid Interface Analysis: Multivalent Hydrophobicity

## Publication Reference
This notebook is the official computational pipeline for the analysis presented in:

> **Modulating the DNA/Lipid Interface through Multivalent Hydrophobicity**  
> Siu Ho Wong et al.  
> *Nano Letters* 2024, **24** (36), 11210–11218  
> DOI: [10.1021/acs.nanolett.4c02564](https://doi.org/10.1021/acs.nanolett.4c02564)

## Scientific Context
This pipeline quantifies the binding of DNA nanostructures to Giant Unilamellar Vesicles (GUVs) via multivalent hydrophobic anchoring. Confocal images contain two channels: the **NBD lipid channel** (used for membrane segmentation) and the **Cy5 DNA channel** (used for fluorescence quantification).

## Pipeline Stages
1. **Preprocessing** — Gaussian smoothing of the lipid channel to reduce noise before thresholding.
2. **Segmentation** — Multi-population Otsu thresholding + binary closing to binarize membrane contours.
3. **Labelling & Edge Exclusion** — Connected-components labelling; vesicles touching image borders are removed.
4. **Clump Exclusion (QC)** — Touching/aggregated vesicles detected via dilation and excluded.
5. **Skeletonization** — Morphological skeletonization followed by controlled dilation to generate a thin membrane mask.
6. **Circularity & Size Filtering (QC)** — Objects below `min_perimeter` or outside the circularity range are discarded.
7. **Quantification** — Background-corrected mean and max Cy5 intensity extracted per vesicle along the membrane mask.

## QC Filter Summary
| Filter | Parameter | Rationale |
|---|---|---|
| Minimum perimeter | `MIN_PERIMETER = 240 px` | Excludes debris and sub-resolution objects |
| Circularity range | `MIN_CIRCULARITY / MAX_CIRCULARITY` | Retains only well-formed spherical vesicles |
| Edge exclusion | automatic | Removes partially imaged vesicles at frame borders |
| Clump exclusion | automatic | Removes aggregated/touching vesicles |

## Equation Labels
- `mean DNA intensity` — raw mean Cy5 intensity along skeletonized membrane mask per vesicle.
- `mean DNA intensity smoothed` — same measurement using Gaussian-smoothed Cy5 channel (σ = 2 px); used for noise-robust comparisons.
- `max DNA intensity` — peak Cy5 pixel value per vesicle membrane mask.
- `out_of_focus` — perimeter / area ratio; higher values indicate out-of-focus vesicles (filter threshold available via `THRESHOLD_OUT_OF_FOCUS`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.filters import gaussian, threshold_multiotsu
from skimage import morphology, measure
from skimage.segmentation import watershed
from scipy import ndimage
import pyclesperanto_prototype as cle
from napari_simpleitk_image_processing import touching_objects_labeling

## Configuration
Set all parameters here before running the pipeline.

In [ ]:
# =============================================================================
# PATHS
# =============================================================================
IMAGE_FOLDER = os.path.join(os.getcwd(), "data", "raw")
OUTPUT_FOLDER = os.path.join(os.getcwd(), "data", "processed")
FILE_OUTPUT = os.path.join(OUTPUT_FOLDER, "Vesicle_Analysis_Results.csv")

# =============================================================================
# SUBFOLDER FILTER
# Set the substring that must appear in a subfolder name for it to be processed.
# Examples: "-d84" for HW210 experiment, "TP" for HW212 experiment.
# =============================================================================
SUBFOLDER_FILTER = "-d84"

# =============================================================================
# SEGMENTATION PARAMETERS
# =============================================================================
SIGMA_LIPID = 1      # Gaussian blur sigma for lipid channel (segmentation)
SIGMA_DNA   = 2      # Gaussian blur sigma for DNA channel (smoothed intensity measurement)
OTSU_CLASSES = 3     # Number of populations for multi-Otsu thresholding
CLOSING_RADIUS = 2   # Disk radius for binary closing (fills small gaps in membrane contour)
DILATION_RADIUS = 3  # Disk radius for skeleton dilation (controls membrane mask thickness)

# =============================================================================
# QC FILTER THRESHOLDS
# =============================================================================
MIN_PERIMETER          = 240    # pixels; excludes small debris
MIN_CIRCULARITY        = 0.2    # 4*pi*area / perimeter^2; lower bound
MAX_CIRCULARITY        = 1.0    # upper bound (1.0 = perfect circle)
THRESHOLD_OUT_OF_FOCUS = 0.5    # perimeter/area ratio; commented out by default (see below)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Input folder : {IMAGE_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Subfolder filter: '{SUBFOLDER_FILTER}'")

## Helper Function

In [ ]:
def watershed_skeletonization(image, mask):
    """Generate a watershed-based skeleton restricted to the membrane mask.
    
    Uses scikit-image watershed with watershed_line=True to find inter-object
    boundaries, then extracts the watershed lines (w==0) as the skeleton.
    Returns a boolean mask of skeleton pixels.
    """
    w = watershed(image, mask=mask, watershed_line=True)
    watershed_skelet = (w == 0) * image * mask
    return watershed_skelet > 0

## Main Analysis Pipeline

In [ ]:
if not os.path.exists(IMAGE_FOLDER):
    print(f"ERROR: Input folder not found: {IMAGE_FOLDER}")
    print("Please create /data/raw/ and add subfolders containing .ome.tif files.")
else:
    image_counter = 0

    for library in sorted(os.listdir(IMAGE_FOLDER)):
        lib_path = os.path.join(IMAGE_FOLDER, library)

        if not (os.path.isdir(lib_path) and SUBFOLDER_FILTER in library):
            continue

        nr = 0
        for image_filename in sorted(os.listdir(lib_path)):
            if "ome.tif" not in image_filename:
                continue

            nr += 1
            image_counter += 1
            full_img_path = os.path.join(lib_path, image_filename)
            print(f"\nProcessing image {image_counter}: {library}/{image_filename}")

            # ------------------------------------------------------------------
            # 1. LOAD CHANNELS
            # ------------------------------------------------------------------
            image_multi = imread(full_img_path)
            img_lipid = image_multi[0].copy()   # Channel 0: NBD lipid
            img_dna   = image_multi[1].copy()   # Channel 1: Cy5 DNA marker

            # ------------------------------------------------------------------
            # 2. PREPROCESSING: Gaussian smoothing
            # ------------------------------------------------------------------
            blurred_lipid = gaussian(img_lipid, sigma=SIGMA_LIPID, preserve_range=True)
            blurred_dna   = gaussian(img_dna,   sigma=SIGMA_DNA,   preserve_range=True)

            # ------------------------------------------------------------------
            # 3. SEGMENTATION: Multi-Otsu binarization + binary closing
            # ------------------------------------------------------------------
            try:
                thresholds = threshold_multiotsu(blurred_lipid, classes=OTSU_CLASSES)
            except Exception as e:
                print(f"  Skipping: Multi-Otsu failed ({e})")
                continue

            binary_image = blurred_lipid > thresholds[0]
            kernel = morphology.disk(CLOSING_RADIUS)
            closed_image = morphology.binary_closing(binary_image, kernel)

            # ------------------------------------------------------------------
            # 4. LABELLING & EDGE EXCLUSION
            # ------------------------------------------------------------------
            labeled_image      = cle.connected_components_labeling_box(closed_image)
            labeled_edges_image = cle.exclude_labels_on_edges(labeled_image)

            mask            = np.array(labeled_edges_image) > 0
            mask_fill_holes = ndimage.binary_fill_holes(mask).astype(int)

            touching_filled_labels = touching_objects_labeling(mask_fill_holes)
            # touching_boarders: label map restricted to membrane contour pixels only
            touching_boarders = touching_filled_labels * mask

            # ------------------------------------------------------------------
            # 5. CLUMP EXCLUSION: detect and remove touching/aggregated vesicles
            # ------------------------------------------------------------------
            minimal_square       = morphology.square(3)
            dilated_filled       = morphology.dilation(touching_filled_labels, minimal_square)
            dilated_object       = mask_fill_holes * dilated_filled
            flag_touching_object = dilated_object != touching_filled_labels

            array_touching_objects = np.empty(1)
            if np.sum(flag_touching_object) > 0:
                array_touching_objects = np.concatenate((
                    dilated_object[flag_touching_object],
                    touching_filled_labels[flag_touching_object]
                ))
                for lbl in array_touching_objects:
                    touching_boarders[touching_boarders == lbl]           = 0
                    touching_filled_labels[touching_filled_labels == lbl] = 0

            # ------------------------------------------------------------------
            # 6. SKELETONIZATION
            # ------------------------------------------------------------------
            skelet = morphology.skeletonize(mask)

            # ------------------------------------------------------------------
            # 7. PRELIMINARY QC: circularity and NaN/inf filtering
            # Circularity = 4π × area_filled / perimeter_contour²
            # (filled area gives stable area; contour perimeter avoids fill artefacts)
            # ------------------------------------------------------------------
            props_filled  = measure.regionprops_table(
                np.array(touching_filled_labels), intensity_image=img_lipid,
                properties=('label', 'area', 'perimeter')
            )
            props_contour = measure.regionprops_table(
                np.array(touching_boarders), intensity_image=img_lipid,
                properties=('label', 'area', 'perimeter', 'mean_intensity')
            )

            circularity = (
                props_filled['area'] * 4 * np.pi / props_contour['perimeter'] ** 2
            )

            # Removed 'median_intensity'; df now built only from available columns.
            df = pd.DataFrame(props_contour)
            df['circularity']   = circularity
            df['image_number']  = image_counter
            print(f"  Before QC filtering: {df.shape[0]} objects")

            # Remove clumped objects
            if np.sum(flag_touching_object) > 0:
                df = df[~np.isin(df.label, array_touching_objects)]

            # Remove NaN / inf rows
            df = df[~df.isin([np.nan, np.inf, -np.inf]).any(axis=1)]

            # Apply circularity filter
            df = df[
                (df.circularity >= MIN_CIRCULARITY) &
                (df.circularity <= MAX_CIRCULARITY)
            ]
            print(f"  After circularity filter: {df.shape[0]} objects")

            # Propagate filtered labels back to image arrays
            filtered_labels = df.label
            touching_boarders[~np.isin(touching_boarders, filtered_labels)]               = 0
            touching_filled_labels[~np.isin(touching_filled_labels, filtered_labels)]     = 0

            # ------------------------------------------------------------------
            # 8. SKELETON DILATION → final membrane mask
            # ------------------------------------------------------------------
            dilated_skelet    = morphology.binary_dilation(skelet, morphology.disk(DILATION_RADIUS))
            touching_boarders = touching_boarders * dilated_skelet

            # ------------------------------------------------------------------
            # 9. FINAL MEASUREMENTS on skeletonized membrane mask
            # ------------------------------------------------------------------
            props_final    = measure.regionprops_table(
                np.array(touching_boarders), intensity_image=img_lipid,
                properties=('label', 'area', 'perimeter', 'mean_intensity', 'centroid', 'max_intensity')
            )
            props_dna      = measure.regionprops_table(
                np.array(touching_boarders), intensity_image=img_dna,
                properties=('label', 'mean_intensity', 'max_intensity')
            )
            props_dna_blur = measure.regionprops_table(
                np.array(touching_boarders), intensity_image=blurred_dna,
                properties=('label', 'mean_intensity')
            )

            # Recalculate circularity on final membrane mask
            props_filled_final  = measure.regionprops_table(
                np.array(touching_filled_labels), intensity_image=img_lipid,
                properties=('label', 'area', 'perimeter')
            )
            props_contour_final = measure.regionprops_table(
                np.array(touching_boarders), intensity_image=img_lipid,
                properties=('label', 'area', 'perimeter')
            )
            circularity_final = (
                props_filled_final['area'] * 4 * np.pi / props_contour_final['perimeter'] ** 2
            )

            # ------------------------------------------------------------------
            # 10. BUILD FINAL DATAFRAME
            # Now built cleanly from props dict, with extra columns added explicitly.
            # ------------------------------------------------------------------
            df_final = pd.DataFrame(props_final)
            df_final.rename(columns={
                'mean_intensity': 'mean_lipid_intensity',
                'max_intensity':  'max_lipid_intensity'
            }, inplace=True)

            df_final['mean_dna_intensity']          = props_dna['mean_intensity']
            df_final['max_dna_intensity']           = props_dna['max_intensity']
            df_final['mean_dna_intensity_smoothed'] = props_dna_blur['mean_intensity']

            df_final['circularity']    = circularity_final
            df_final['out_of_focus']   = df_final['perimeter'] / df_final['area']
            df_final['image_number']   = image_counter
            df_final['file_name']      = image_filename[:-4]
            df_final['library']        = library
            df_final['nr_in_library']  = nr

            # Size filter: remove objects below minimum perimeter
            df_final = df_final[df_final.perimeter >= MIN_PERIMETER]
            print(f"  After size filter: {df_final.shape[0]} vesicles retained")

            # Out-of-focus filter (disabled by default; uncomment to activate)
            # df_final = df_final[df_final.out_of_focus <= THRESHOLD_OUT_OF_FOCUS]

            # Sync label arrays with final accepted vesicles
            final_labels = df_final.label
            labeled_skelet = skelet * touching_boarders
            labeled_skelet[~np.isin(labeled_skelet, final_labels)]       = 0
            touching_boarders[~np.isin(touching_boarders, final_labels)] = 0

            # ------------------------------------------------------------------
            # 11. SAVE RESULTS (append mode after first image)
            # ------------------------------------------------------------------
            write_header = (image_counter == 1)
            df_final.to_csv(
                FILE_OUTPUT,
                mode='w' if write_header else 'a',
                index=True,
                header=write_header
            )

            # ------------------------------------------------------------------
            # 12. SAVE DIAGNOSTIC FIGURE (4-panel)
            # ------------------------------------------------------------------
            fig, ax = plt.subplots(2, 2, figsize=(10, 10))

            ax[0, 0].imshow(img_lipid, cmap='gray')
            ax[0, 0].set_title('NBD lipid channel (raw)')

            ax[0, 1].imshow(labeled_edges_image, cmap='gray')
            ax[0, 1].set_title('After edge exclusion')

            ax[1, 0].imshow(touching_boarders > 0, cmap='gray')
            ax[1, 0].set_title('Accepted vesicle masks')

            ax[1, 1].imshow(img_lipid, cmap='gray')
            ax[1, 1].imshow(
                touching_boarders,
                cmap='jet',
                alpha=0.5 * (touching_boarders > 0)
            )
            ax[1, 1].set_title('Segmentation overlay')
            for _, row in df_final.iterrows():
                ax[1, 1].text(
                    int(row['centroid-1']), int(row['centroid-0']),
                    str(int(row['label'])), fontsize=5, color='red'
                )

            fig_path = os.path.join(
                OUTPUT_FOLDER,
                f"{library}_{image_filename[:-4]}_segmentation.tif"
            )
            fig.savefig(fig_path, dpi=300)
            plt.close(fig)

    print(f"\n✓ Pipeline complete. {image_counter} images processed.")
    print(f"  Results saved to: {FILE_OUTPUT}")

## Results Preview

In [ ]:
if os.path.exists(FILE_OUTPUT):
    df_results = pd.read_csv(FILE_OUTPUT, index_col=0)
    print(f"Total vesicles analysed: {len(df_results)}")
    print(f"Libraries: {df_results['library'].unique()}")
    display(df_results[['library', 'file_name', 'mean_dna_intensity',
                         'mean_dna_intensity_smoothed', 'circularity',
                         'out_of_focus']].head(10))
else:
    print("No results file found. Run the pipeline cell above first.")